# Baseline 1 — Independent DINOv2 ViT-B/14 classifiers

Two separate DINOv2 ViT-B/14 models predict crop and disease. Their public
pretrained backbones are frozen; the classifier heads learn from DPD-Small.
Candidate pair score = `log P(crop) + log P(disease)`.

Run from the `baselines/` directory of the full repository, with `common/`
available one level above. Development reads only `train.csv` and its 323
training-derived combinations. The official ten unseen combinations are
introduced only in the optional final-evaluation cell after selection is
frozen. Running all cells with the defaults does not read official tests.

This is a head-only adaptation for limited GPU memory, not full DINOv2
fine-tuning. The published checkpoint and official exploratory results are in
the `baseline1-dinov2-base-frozen-v1` GitHub Release.


In [ ]:
import copy
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / 'common').is_dir() else Path.cwd().parent
if not (REPO_ROOT / 'common').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or baselines/ folder.')
sys.path.insert(0, str(REPO_ROOT))

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import config
from common.dataset import get_loaders, get_pairs, get_final_test_loaders
from common.evaluate import val_accuracy, evaluate

MODEL_NAME = 'baseline1_dinov2_base_frozen'
BACKBONE = 'vit_base_patch14_dinov2.lvd142m'
PROTOCOL = 'training_only_candidates_dinov2_base_frozen_v1'
DEBUG = False
RUN_TRAINING = True
RUN_FINAL_EVALUATION = False  # Change only after all model choices are frozen.
DATA_ROOT = None  # e.g. Path(r'C:\Users\User\Downloads\DPD_Small'); None uses repo/data

if DATA_ROOT is not None:
    config.DATA_DIR = Path(DATA_ROOT)
    config.IMAGE_DIR = config.DATA_DIR / 'DPD_Small_dataset'
    config.TRAIN_CSV = config.DATA_DIR / 'DPD_Small_train.csv'
    config.TEST_SEEN_CSV = config.DATA_DIR / 'DPD_Small_test_seen.csv'
    config.TEST_UNSEEN_CSV = config.DATA_DIR / 'DPD_Small_test_unseen.csv'

HP = {'backbone': BACKBONE, 'input_size': 224, 'backbone_frozen': True,
      'batch_size': 8 if DEBUG else 16, 'epochs': 1 if DEBUG else 20,
      'crop_lr': 1e-4, 'disease_lr': 3e-4, 'weight_decay': 1e-4,
      'label_smoothing': 0.1, 'debug': DEBUG, 'protocol': PROTOCOL}
RUN_NAME = MODEL_NAME + ('_debug' if DEBUG else '')
config.NUM_WORKERS = 0  # Avoid Windows data-worker memory errors.
config.set_seed(42)
DEVICE = config.DEVICE
CHECKPOINT_PATH = config.CHECKPOINT_DIR / f'{RUN_NAME}.pt'
print('Device:', DEVICE, 'Data:', config.DATA_DIR)


## Model and training-only data

`timm` resamples DINOv2 positional embeddings from its default 518-pixel grid
for the project's 224-pixel images. Both models use the same cached public
pretrained weights, with separate crop and disease heads.


In [ ]:
def make_model(classes, pretrained):
    model = timm.create_model(BACKBONE, pretrained=pretrained,
                              num_classes=classes, img_size=HP['input_size'])
    model.requires_grad_(False)
    model.head.requires_grad_(True)
    return model


@torch.no_grad()
def pair_scores(crop_model, disease_model, images, pairs):
    crop_model.eval()
    disease_model.eval()
    images = images.to(DEVICE)
    crop_log = F.log_softmax(crop_model(images).float(), dim=1)
    disease_log = F.log_softmax(disease_model(images).float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


if RUN_TRAINING:
    train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
        HP['batch_size'], norm='imagenet', debug=DEBUG)
    assert seen_loader is None and unseen_loader is None
    train_loader.pin_memory = False
    val_loader.pin_memory = False
    development_pairs = get_pairs().to(DEVICE)
    assert development_pairs.shape == (323, 2)
    crop_model = make_model(config.NUM_CROPS, True).to(DEVICE)
    disease_model = make_model(config.NUM_DISEASES, True).to(DEVICE)


## Train and select the best epoch

Each head has its own AdamW optimizer and cosine learning-rate schedule.
The best epoch is chosen by validation Top-1 on training-derived pairs only.
No official test labels or unseen candidate identities are used here.


In [ ]:
if RUN_TRAINING:
    crop_opt = torch.optim.AdamW(crop_model.head.parameters(),
                                 lr=HP['crop_lr'], weight_decay=HP['weight_decay'])
    disease_opt = torch.optim.AdamW(disease_model.head.parameters(),
                                    lr=HP['disease_lr'], weight_decay=HP['weight_decay'])
    crop_schedule = torch.optim.lr_scheduler.CosineAnnealingLR(crop_opt, T_max=HP['epochs'])
    disease_schedule = torch.optim.lr_scheduler.CosineAnnealingLR(disease_opt, T_max=HP['epochs'])
    criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
    best_val, best_state = -1.0, None

    for epoch in range(HP['epochs']):
        crop_model.eval(); disease_model.eval()  # frozen backbones stay in eval mode
        crop_model.head.train(); disease_model.head.train()
        for images, crops, diseases, _ in train_loader:
            images = images.to(DEVICE)
            crops, diseases = crops.to(DEVICE), diseases.to(DEVICE)
            crop_opt.zero_grad(set_to_none=True)
            crop_loss = criterion(crop_model(images), crops)
            crop_loss.backward(); crop_opt.step()
            disease_opt.zero_grad(set_to_none=True)
            disease_loss = criterion(disease_model(images), diseases)
            disease_loss.backward(); disease_opt.step()

        crop_schedule.step(); disease_schedule.step()
        top1 = val_accuracy(lambda x: pair_scores(crop_model, disease_model,
                                                   x, development_pairs), val_loader)
        print(f"epoch {epoch + 1}/{HP['epochs']}: crop loss {crop_loss.item():.3f}, "
              f"disease loss {disease_loss.item():.3f}, val top-1 {top1:.2f}%", flush=True)
        if top1 > best_val:
            best_val = top1
            best_state = {'crop': copy.deepcopy(crop_model.state_dict()),
                          'disease': copy.deepcopy(disease_model.state_dict())}

    config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save({'model_name': RUN_NAME, 'state_dict': best_state, 'hp': HP,
                'best_validation_top1': best_val}, CHECKPOINT_PATH)
    print('Saved:', CHECKPOINT_PATH)
    print('Official tests require the separate final-evaluation cell.')


## Optional final evaluation

Set `RUN_FINAL_EVALUATION = True` in the first code cell **only after** all
development choices are frozen. Set `RUN_TRAINING = False` to load an existing
checkpoint without retraining. The checkpoint must have this notebook's
protocol marker and must not be a debug checkpoint. The model and score rule
are loaded before the official test CSVs are opened. Do not tune after viewing
these test results; previous project experiments have already inspected the
official test set, so report these results as exploratory.


In [ ]:
if RUN_FINAL_EVALUATION:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
    hp = checkpoint['hp']
    if hp.get('protocol') != PROTOCOL or hp.get('debug') or hp.get('backbone') != BACKBONE:
        raise ValueError('Expected a full DINOv2 ViT-B/14 checkpoint from this notebook.')
    crop_model = make_model(config.NUM_CROPS, False)
    disease_model = make_model(config.NUM_DISEASES, False)
    crop_model.load_state_dict(checkpoint['state_dict']['crop'])
    disease_model.load_state_dict(checkpoint['state_dict']['disease'])
    crop_model.to(DEVICE).eval(); disease_model.to(DEVICE).eval()
    final_pairs = get_pairs(final_evaluation=True).to(DEVICE)
    seen_loader, unseen_loader = get_final_test_loaders(
        hp['batch_size'], norm='imagenet', final_evaluation=True)
    results = evaluate(lambda x: pair_scores(crop_model, disease_model, x, final_pairs),
                       seen_loader, unseen_loader,
                       checkpoint['model_name'] + '_final', hp, robust=True)
